In [1]:
from semantic_agent import SemanticAgent
import pandas as pd

agent = SemanticAgent()

## Q1 "How many loads were delivered in the last full month available in the data?"

In [2]:
q1 = "How many loads were delivered in the last full month available in the data?"
result_q1 = agent.answer_question(q1)

print(f"Question: {result_q1['question']}")
print(f"\nSQL:\n{result_q1['generated_sql']}")
print(f"\nAnswer: {result_q1['answer']}")
print(f"Correct? {'Yes' if result_q1['correct'] else 'No'}")

Question: How many loads were delivered in the last full month available in the data?

SQL:
WITH last_month AS (
    SELECT date_trunc('month', MAX(delivery_date)) - INTERVAL '1 month' AS month_start
    FROM analytics.fact_loads
)
SELECT COUNT(*) AS loads_delivered
FROM analytics.fact_loads, last_month
WHERE delivery_date >= last_month.month_start
  AND delivery_date < last_month.month_start + INTERVAL '1 month';

Answer: [{'loads_delivered': 0}]
Correct? Yes


**Notes:** Ran without a SQL error, but the answer is misleading. The query defines "last full month" as `date_trunc('month', MAX(delivery_date)) - 1 month`. Since `MAX(delivery_date)` is 2025-03-15, that resolves to February 2025 — which has **zero** rows in the data (delivery volume falls off a cliff after Dec 2024: 503/528/497 loads/month Oct-Dec 2024, then just 65 in Jan 2025, 0 in Feb 2025, and a single stray record on 2025-03-15). So the literal answer (0) is technically what the SQL computes, but it isn't a useful business answer — it reflects a data gap, not zero deliveries. The *meaningful* last full month by representative volume is **December 2024 (442 delivered, i.e. not cancelled)**. The query also doesn't exclude `load_was_cancelled`, though that's moot here since the result set is empty. This row also shows a limitation of the agent's own `correct?` flag — it only checks "did the SQL execute," not "is the answer sensible."

## Q2 "Which shipper had the highest total book price?"

In [3]:
q2 = "Which shipper had the highest total book price?"
result_q2 = agent.answer_question(q2)

print(f"Question: {result_q2['question']}")
print(f"\nSQL:\n{result_q2['generated_sql']}")
print(f"\nAnswer: {result_q2['answer']}")
print(f"Correct? {'✓ Yes' if result_q2['correct'] else '✗ No'}")

Question: Which shipper had the highest total book price?

SQL:
SELECT 
    ds.shipper_name,
    SUM(fl.book_price) AS total_book_price
FROM analytics.fact_loads fl
JOIN analytics.dim_shipper ds
    ON fl.shipper_id = ds.shipper_id
GROUP BY ds.shipper_name
ORDER BY total_book_price DESC
LIMIT 1;

Answer: [{'shipper_name': 'Shipper 1249', 'total_book_price': 1915694.1600000034}]
Correct? ✓ Yes


**Notes:** **Correct.** Ran successfully on the first try and matches an independently verified ground-truth query (direct join `fact_loads` -> `dim_shipper`, `SUM(book_price)` grouped by shipper, no filter on delivered/cancelled since the question just says "total book price"). Ground truth: Shipper 1249, ~$1,915,694.16.

## Q3 "What is the average book price per load by pickup state?"

In [4]:
q3 = "What is the average book price per load by pickup state?"
result_q3 = agent.answer_question(q3)

print(f"Question: {result_q3['question']}")
print(f"\nSQL:\n{result_q3['generated_sql']}")
print(f"\nAnswer: {result_q3['answer']}")
print(f"Correct? {'✓ Yes' if result_q3['correct'] else '✗ No'}")

Question: What is the average book price per load by pickup state?

SQL:
SELECT
    dim_location.state AS pickup_state,
    AVG(analytics.fact_loads.book_price) AS avg_book_price_per_load
FROM analytics.fact_loads
JOIN analytics.dim_location
    ON analytics.fact_loads.pickup_location_id = dim_location.location_id
GROUP BY dim_location.state
ORDER BY avg_book_price_per_load DESC;

Answer: None
Correct? ✗ No


**Notes:** Incorrect. The agent invented a column `pickup_location_id` directly on `fact_loads`, which does not exist — pickup location is only reachable via `fact_loads.lane_id -> dim_lane.source_location_id -> dim_location.location_id`. The schema text built from the dbt manifest lists each mart's columns independently without describing this join path, so the model falls back to a flat-schema assumption (it correctly guessed the final column name, `dim_location.state`, but not how to get there). Ground truth (verified manually): highest avg book price by pickup state is SC (~$2,708.99, n=88).

## Q4 "What are the top 5 lanes by number of delivered loads?"

In [5]:
q4 = "What are the top 5 lanes by number of delivered loads?"
result_q4 = agent.answer_question(q4)

print(f"Question: {result_q4['question']}")
print(f"\nSQL:\n{result_q4['generated_sql']}")
print(f"\nAnswer: {result_q4['answer']}")
print(f"Correct? {'✓ Yes' if result_q4['correct'] else '✗ No'}")

Question: What are the top 5 lanes by number of delivered loads?

SQL:
SELECT
    origin_city,
    destination_city,
    COUNT(*) AS delivered_load_count
FROM analytics.fact_loads
WHERE status = 'Delivered'
GROUP BY origin_city, destination_city
ORDER BY delivered_load_count DESC
LIMIT 5;

Answer: None
Correct? ✗ No


**Notes:** **Incorrect.** The agent assumed `origin_city`/`origin_state`/`destination_city`/`destination_state` columns and a `status` column on `fact_loads`, none of which exist. Lane identity lives on `dim_lane.lane_name` (format "city,ST -> city,ST"); there is no status column, "delivered" should be inferred as `NOT load_was_cancelled`. Ground truth top lane: Hawkins,TX -> Roanoke,TX with 882 delivered loads — a clear outlier vs. #2 at 150.

## Q5 "Which carrier moved the most loads into Texas?"

In [6]:
q5 = "Which carrier moved the most loads into Texas?"
result_q5 = agent.answer_question(q5)

print(f"Question: {result_q5['question']}")
print(f"\nSQL:\n{result_q5['generated_sql']}")
print(f"\nAnswer: {result_q5['answer']}")
print(f"Correct? {'✓ Yes' if result_q5['correct'] else '✗ No'}")

Question: Which carrier moved the most loads into Texas?

SQL:
SELECT
    dc.carrier_name,
    COUNT(*) AS load_count
FROM analytics.fact_loads fl
JOIN analytics.dim_carrier dc
    ON fl.carrier_id = dc.carrier_id
WHERE fl.dest_state = 'TX'
GROUP BY dc.carrier_name
ORDER BY load_count DESC
LIMIT 1;

Answer: None
Correct? ✗ No


**Notes:** Incorrect. Used `fl.dest_state`, which doesn't exist on `fact_loads`. Correct path is `fact_loads.lane_id -> dim_lane.target_location_id -> dim_location.state = 'TX'`. Ground truth: Carrier 567581 moved the most loads into TX (188 loads).

## Q6 "How does the average book price compare between intrastate and interstate loads?"

In [7]:
q6 = "How does the average book price compare between intrastate and interstate loads?"
result_q6 = agent.answer_question(q6)

print(f"Question: {result_q6['question']}")
print(f"\nSQL:\n{result_q6['generated_sql']}")
print(f"\nAnswer: {result_q6['answer']}")
print(f"Correct? {'✓ Yes' if result_q6['correct'] else '✗ No'}")

Question: How does the average book price compare between intrastate and interstate loads?

SQL:
SELECT
    CASE
        WHEN origin_state = dest_state THEN 'Intrastate'
        ELSE 'Interstate'
    END AS load_type,
    AVG(book_price) AS avg_book_price,
    COUNT(*) AS load_count
FROM analytics.fact_loads
GROUP BY
    CASE
        WHEN origin_state = dest_state THEN 'Intrastate'
        ELSE 'Interstate'
    END;

Answer: None
Correct? ✗ No


**Notes:** Incorrect. Same root cause as Q4/Q5 — assumed `origin_state`/`dest_state` directly on `fact_loads`. Needs two joins to `dim_location` (once via `dim_lane.source_location_id`, once via `dim_lane.target_location_id`) to compare states. Ground truth: Interstate loads average ~$1,709.82 (n=3,534) vs. Intrastate ~$584.49 (n=1,821) — interstate loads cost ~2.9x more on average, consistent with longer average mileage.

## Q7 "For the shipper with the most delivered loads, how did monthly volume change across the period covered by the data?"

In [8]:
q7 = "For the shipper with the most delivered loads, how did monthly volume change across the period covered by the data?"
result_q7 = agent.answer_question(q7)

print(f"Question: {result_q7['question']}")
print(f"\nSQL:\n{result_q7['generated_sql']}")
print(f"\nAnswer: {result_q7['answer']}")
print(f"Correct? {'✓ Yes' if result_q7['correct'] else '✗ No'}")

Question: For the shipper with the most delivered loads, how did monthly volume change across the period covered by the data?

SQL:
WITH top_shipper AS (
    SELECT
        fl.shipper_id,
        COUNT(*) AS delivered_loads
    FROM analytics.fact_loads fl
    WHERE fl.load_status = 'DELIVERED'
    GROUP BY fl.shipper_id
    ORDER BY delivered_loads DESC
    LIMIT 1
)
SELECT
    DATE_TRUNC('month', fl.delivery_date) AS month,
    COUNT(*) AS monthly_delivered_loads
FROM analytics.fact_loads fl
JOIN top_shipper ts
    ON fl.shipper_id = ts.shipper_id
WHERE fl.load_status = 'DELIVERED'
GROUP BY DATE_TRUNC('month', fl.delivery_date)
ORDER BY month;

Answer: None
Correct? ✗ No


**Notes:** Incorrect, though the closest to correct of the failing queries — the CTE structure and monthly `DATE_TRUNC` grouping are sound. It fails because it filters on a non-existent `fact_loads.load_status = 'DELIVERED'`; should be `NOT load_was_cancelled`. It also drops the `dim_shipper` join (no shipper name in the output) once inside the CTE, which is a minor readability miss rather than a correctness one. Ground truth top shipper by delivered load count: Shipper 758 (1,834 delivered loads), well ahead of Shipper 1249 (1,101).

## Q8 "Among lanes with at least 10 delivered loads, which had the highest average book price?"

In [9]:
q8 = "Among lanes with at least 10 delivered loads, which had the highest average book price?"
result_q8 = agent.answer_question(q8)

print(f"Question: {result_q8['question']}")
print(f"\nSQL:\n{result_q8['generated_sql']}")
print(f"\nAnswer: {result_q8['answer']}")
print(f"Correct? {'✓ Yes' if result_q8['correct'] else '✗ No'}")

Question: Among lanes with at least 10 delivered loads, which had the highest average book price?

SQL:
SELECT
    fact_loads.lane_id,
    dim_lane.origin,
    dim_lane.destination,
    COUNT(*) AS delivered_load_count,
    AVG(fact_loads.book_price) AS avg_book_price
FROM analytics.fact_loads
JOIN analytics.dim_lane
    ON fact_loads.lane_id = dim_lane.lane_id
WHERE fact_loads.load_status = 'Delivered'
GROUP BY
    fact_loads.lane_id,
    dim

Answer: None
Correct? ✗ No


**Notes:** Incorrect, and it surfaces a second, distinct bug on top of the usual hallucination. It invents `fact_loads.load_status` and `dim_lane.origin`/`dim_lane.destination` (the real column is `dim_lane.lane_name`) — same recurring failure mode as Q4/Q5/Q7. But the query is also **truncated mid-statement** ("...GROUP BY\n    fact_loads.lane_id,\n    dim"), because `generate_sql()` in `semantic_agent.py` caps `max_tokens=500`, which isn't enough for a longer multi-column query. That's a configuration bug in the agent itself, independent of schema-grounding — worth raising `max_tokens` (or streaming/retrying on a stop reason of `max_tokens`) regardless of the column-naming fix. Ground truth top lane (>=10 delivered loads): Stockton,CA -> Parrish,FL at a flat $6,800.00 avg (n=11, likely a fixed contracted rate).

## Summary

In [27]:
results_all = [result_q1, result_q2, result_q3, result_q4, result_q5, result_q6, result_q7, result_q8]

manual_notes = [
    "SQL runs fine, but answer (0) is misleading: literal 'last month before max(delivery_date)' = Feb 2025, which has no data due to a volume drop-off after Dec 2024. See notes above.",
    "Correct — matches ground truth; simple join + SUM, no hallucinated columns.",
    "Incorrect — hallucinated `pickup_location_id` on fact_loads; needs join through dim_lane.source_location_id -> dim_location.",
    "Incorrect — hallucinated origin/destination city/state columns and a `status` column; lane info lives on dim_lane.lane_name.",
    "Incorrect — hallucinated `dest_state`; needs dim_lane.target_location_id -> dim_location.state = 'TX'.",
    "Incorrect — hallucinated origin_state/dest_state; needs two joins to dim_location (source and target).",
    "Incorrect — hallucinated `load_status = 'DELIVERED'`; should filter on NOT load_was_cancelled. CTE/grouping logic otherwise sound.",
    "Incorrect — hallucinated `load_status`/`dim_lane.origin`/`dim_lane.destination`, AND the SQL is truncated mid-query (max_tokens=500 too low).",
]

summary_df = pd.DataFrame({
    "question": [r["question"] for r in results_all],
    "generated_sql": [r["generated_sql"] for r in results_all],
    "answer": [r["answer"] for r in results_all],
    "correct?": [r["correct"] for r in results_all],
    "notes": manual_notes,
})
summary_df

                                            question  ...                                              notes
0  How many loads were delivered in the last full...  ...  SQL runs fine, but answer (0) is misleading: l...
1    Which shipper had the highest total book price?  ...  Correct — matches ground truth; simple join + ...
2  What is the average book price per load by pic...  ...  Incorrect — hallucinated `pickup_location_id` ...
3  What are the top 5 lanes by number of delivere...  ...  Incorrect — hallucinated origin/destination ci...
4     Which carrier moved the most loads into Texas?  ...  Incorrect — hallucinated `dest_state`; needs d...
5  How does the average book price compare betwee...  ...  Incorrect — hallucinated origin_state/dest_sta...
6  For the shipper with the most delivered loads,...  ...  Incorrect — hallucinated `load_status = 'DELIV...
7  Among lanes with at least 10 delivered loads, ...  ...  Incorrect — hallucinated `load_status`/`dim_la...

[8 rows x 5 column